In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

from pathlib import Path  
import glob

In [ ]:
# directory_path = "/Users/<user>/Downloads/New v copy/src/data/earnings_calls"
directory_path = "/Users/<user>/Downloads/Diliya Work/src/data/earnings_calls/2016"

In [ ]:
text_files = glob.glob(f"{directory_path}/*.xml")
text_files

In [ ]:
text_titles = [Path(text).stem for text in text_files]
text_titles

In [ ]:
tfidf_vectorizer = TfidfVectorizer(input='filename', stop_words='english',ngram_range=(2,2))
tfidf_vector = tfidf_vectorizer.fit_transform(text_files)
tfidf_df = pd.DataFrame(tfidf_vector.toarray(), index=text_titles, columns=tfidf_vectorizer.get_feature_names_out())

In [ ]:
# tfidf_df.loc['00_Document Frequency'] = (tfidf_df > 0).sum()
# tfidf_slice = tfidf_df[['earning', 'transcript', 'increase', 'conference', 'lost', 'stocks','tax', 'men', 'women', 'children']]
# tfidf_slice.sort_index().round(decimals=2)

In [ ]:
# tfidf_df = tfidf_df.drop('00_Document Frequency', errors='ignore')

In [ ]:

tfidf_df.stack().reset_index()
# tfidf_df = tfidf_df.rename(columns={0:'tfidf', 'level_0': 'document','level_1': 'term', 'level_2': 'term'})
# tfidf_df.sort_values(by=['document','tfidf'], ascending=[True,False]).groupby(['document']).head(10)

In [ ]:
tfidf_df = tfidf_df.stack().reset_index()
tfidf_df = tfidf_df.rename(columns={0:'tfidf', 'level_0': 'document','level_1': 'term', 'level_2': 'term'})

In [ ]:
tfidf_df.sort_values(by=['document','tfidf'], ascending=[True,False]).groupby(['document']).head(10)

In [ ]:
top_tfidf = tfidf_df.sort_values(by=['document','tfidf'], ascending=[True,False]).groupby(['document']).head(10)
top_tfidf[top_tfidf['term'].str.contains('b2b')]

In [ ]:
top_tfidf[top_tfidf['document'].str.contains('7330148')]

In [ ]:
import altair as alt
import numpy as np

# Terms in this list will get a red dot in the visualization
term_list = ['war', 'peace']

# adding a little randomness to break ties in term ranking
top_tfidf_plusRand = top_tfidf.copy()
top_tfidf_plusRand['tfidf'] = top_tfidf_plusRand['tfidf'] + np.random.rand(top_tfidf.shape[0])*0.0001

# base for all visualizations, with rank calculation
base = alt.Chart(top_tfidf_plusRand).encode(
    x = 'rank:O',
    y = 'document:N'
).transform_window(
    rank = "rank()",
    sort = [alt.SortField("tfidf", order="descending")],
    groupby = ["document"],
)

# heatmap specification
heatmap = base.mark_rect().encode(
    color = 'tfidf:Q'
)

# red circle over terms in above list
circle = base.mark_circle(size=100).encode(
    color = alt.condition(
        alt.FieldOneOfPredicate(field='term', oneOf=term_list),
        alt.value('red'),
        alt.value('#FFFFFF00')        
    )
)

# text labels, white for darker heatmap colors
text = base.mark_text(baseline='middle').encode(
    text = 'term:N',
    color = alt.condition(alt.datum.tfidf >= 0.23, alt.value('white'), alt.value('black'))
)

# display the three superimposed visualizations
(heatmap + circle + text).properties(width = 600)